# Phase 3: Model Training + Fairness Evaluation

**Run this notebook on Google Colab (GPU runtime).**

**Goal:** Train the dual-branch ANN, evaluate performance, run a fairness check on sensitive attributes, and save the model.

**Input:**
- `data/processed/train_processed.parquet`
- `models/encoder_objects.pkl` (for cardinalities)

**Output:**
- `models/best_model.keras`

## 0. Colab Setup

Run this cell only on Colab. It clones the repo and mounts your Drive for data access.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# !git clone https://github.com/AayushDeshmukh9090/credit-scoring-multimodal.git
# %cd credit-scoring-multimodal

In [ ]:
# !mkdir -p data/processed models

In [ ]:
# !cp /content/drive/MyDrive/MultiModal_portfolio_project/train_processed.parquet data/processed/
# !cp /content/drive/MyDrive/MultiModal_portfolio_project/test_processed.parquet data/processed/
# !cp /content/drive/MyDrive/MultiModal_portfolio_project/scaler.pkl models/
# !cp /content/drive/MyDrive/MultiModal_portfolio_project/encoder_objects.pkl models/

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle

import tensorflow as tf
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

from src.data_loader import (
    load_processed_train,
    split_features_and_target,
    train_val_split,
    compute_class_weights,
    CATEGORICAL_FEATURES, NUMERICAL_FEATURES
)
from src.model_builder import build_multimodal_model, prepare_model_inputs

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load Data

In [ ]:
df = load_processed_train(path='/content/credit-scoring-multimodal/data/processed/train_processed.parquet')
print(df.shape)
df.head()

In [ ]:
X_num, X_cat, y = split_features_and_target(df)

X_num_tr, X_num_val, X_cat_tr, X_cat_val, y_tr, y_val = train_val_split(
    X_num, X_cat, y, val_size=0.2
)

print(f"Train size: {len(y_tr)} | Val size: {len(y_val)}")
print(f"Train default rate: {y_tr.mean():.3f} | Val default rate: {y_val.mean():.3f}")

## 2. Prepare Model Inputs

In [ ]:
# Load cardinalities from saved encoder objects
with open('/content/credit-scoring-multimodal/models/encoder_objects.pkl', 'rb') as f:
    encoder_payload = pickle.load(f)

cat_cardinalities = encoder_payload['cardinalities']
print("Cardinalities:", cat_cardinalities)

num_feature_count = len(NUMERICAL_FEATURES)
print(f"Numerical features: {num_feature_count}")

In [ ]:
train_inputs = prepare_model_inputs(X_num_tr, X_cat_tr)
val_inputs   = prepare_model_inputs(X_num_val, X_cat_val)

print(f"Total input arrays: {len(train_inputs)}")
print(f"Numerical input shape: {train_inputs[0].shape}")
print(f"First categorical input shape: {train_inputs[1].shape}")

## 3. Build Model

In [ ]:
model = build_multimodal_model(
    num_feature_count=num_feature_count,
    cat_feature_cardinalities=cat_cardinalities,
    learning_rate=0.001
)

model.summary()

## 4. Train

In [ ]:
class_weights = compute_class_weights(y_tr)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc',
    patience=5,
    restore_best_weights=True,
    mode='max'
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc',
    factor=0.5,
    patience=3,
    mode='max',
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    x=train_inputs,
    y=y_tr.to_numpy(),
    epochs=30,
    batch_size=512,
    validation_data=(val_inputs, y_val.to_numpy()),
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# AUC
axes[0].plot(history.history['auc'], label='Train AUC')
axes[0].plot(history.history['val_auc'], label='Val AUC')
axes[0].set_title('AUC over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Loss
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Evaluation on Validation Set

In [ ]:
y_pred_proba = model.predict(val_inputs).flatten()
y_pred_class = (y_pred_proba >= 0.5).astype(int)

val_auc = roc_auc_score(y_val, y_pred_proba)
print(f"Validation AUC: {val_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred_class, target_names=['Repaid', 'Default']))

In [ ]:
cm = confusion_matrix(y_val, y_pred_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Repaid', 'Default'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix (Validation Set)')
plt.show()

## 7. Fairness Evaluation

We check whether the model's predictions differ systematically across gender groups.

**Metric used: Demographic Parity Difference**
- Measures: Does P(predicted default | Male) ≈ P(predicted default | Female)?
- A value close to 0 means the model treats both groups similarly.
- A large positive or negative value indicates potential bias.

This is a real, recognised fairness metric — not a custom penalty term.

In [ ]:
# Reconstruct validation set with sensitive attribute for fairness analysis
# CODE_GENDER in our processed data: check mappings/CODE_GENDER.json for int→label
import json

with open('/content/credit-scoring-multimodal/data/mappings/CODE_GENDER.json') as f:
    gender_mapping = json.load(f)  # e.g. {"0": "F", "1": "M"}

print("Gender mapping:", gender_mapping)

val_gender = X_cat_val['CODE_GENDER'].values  # integer-encoded
unique_groups = sorted(set(val_gender))

print("\n--- Fairness Report: Demographic Parity ---")
group_default_rates = {}

for g in unique_groups:
    mask = val_gender == g
    group_preds = y_pred_class[mask]
    group_proba = y_pred_proba[mask]
    group_actual = y_val.values[mask]
    
    pred_default_rate = group_preds.mean()
    actual_default_rate = group_actual.mean()
    group_auc = roc_auc_score(group_actual, group_proba)
    label = gender_mapping.get(str(g), str(g))
    
    group_default_rates[label] = pred_default_rate
    
    print(f"  Group: {label} (n={mask.sum()})")
    print(f"    Actual default rate:    {actual_default_rate:.4f}")
    print(f"    Predicted default rate: {pred_default_rate:.4f}")
    print(f"    Group AUC:              {group_auc:.4f}")
    print()

In [ ]:
# Demographic Parity Difference
rates = list(group_default_rates.values())
dpd = max(rates) - min(rates)
print(f"Demographic Parity Difference: {dpd:.4f}")
print()
if dpd < 0.05:
    print("PASS: DPD < 0.05 — model predictions are broadly similar across gender groups.")
else:
    print("REVIEW: DPD >= 0.05 — meaningful difference in predicted default rates across gender groups.")
    print("This does not necessarily mean discrimination, but warrants investigation.")

In [ ]:
# Visualise
labels = list(group_default_rates.keys())
values = list(group_default_rates.values())

plt.figure(figsize=(6, 4))
plt.bar(labels, values, color=['steelblue', 'salmon'])
plt.axhline(y=sum(values)/len(values), color='black', linestyle='--', label='Overall avg')
plt.ylabel('Predicted Default Rate')
plt.title('Demographic Parity Check by Gender')
plt.legend()
plt.show()

## 8. Save Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

model.save('../models/best_model.keras')
print("Model saved to models/best_model.keras")

# On Colab: download to local machine
# from google.colab import files
# files.download('../models/best_model.keras')

## Summary

| Metric | Value |
|--------|-------|
| Validation AUC | (fill after training) |
| Demographic Parity Difference | (fill after training) |
| Epochs trained | (fill after training) |

**Next:** Place `best_model.keras` in your local `models/` folder, then build the API.